# GLP-1 Environmental Impact Analysis
**Author:** Marcus Mashanda  
**Project:** Downstream Environmental Consequences of Mass GLP-1 Adoption  
**GitHub:** [marcusmashanda1/environmental-data](https://github.com/marcusmashanda1/environmental-data)

---

## Overview
This notebook models the environmental risk posed by GLP-1 receptor agonists (semaglutide, liraglutide, dulaglutide, exenatide) entering wastewater systems as their global adoption accelerates.

### Methodology
1. **Data Pipeline** — WHO ATC/DDD values + literature-derived pharmacokinetics
2. **MEC Calculation** — Measured Environmental Concentration in WWTP effluent
3. **Risk Quotient (RQ)** — RQ = MEC / PNEC; RQ > 1 indicates ecological risk
4. **Scenario Modeling** — Conservative (2024), Baseline (2027), High Adoption (2030)

### Key References
- WHO ATC/DDD Index (A10BJ), 2026
- Brinch et al. (2020) — semaglutide environmental occurrence & PNEC
- Kling et al. (2022) — GLP-1 WWTP removal efficiency
- Drucker (2022) — GLP-1 pharmacokinetics and excretion pathways

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import project pipeline
import sys
sys.path.append('..')
from src.pipeline import run_pipeline, SCENARIOS

# Plot styling
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('✓ Imports successful')

---
## 2. Run Data Pipeline

In [ ]:
# Run pipeline and load all datasets
results = run_pipeline(output_dir='../data')

drug_props  = results['drug_properties']
daily_loads = results['daily_loads']
mec_inputs  = results['mec_inputs']

---
## 3. Drug Properties

In [ ]:
drug_props[['drug', 'route', 'ddd_mg', 'excretion_fraction', 'wwtp_removal', 'pnec_ng_L']]

---
## 4. MEC & Risk Quotient Summary

In [ ]:
# Risk quotient pivot table
rq_summary = daily_loads.pivot_table(
    index=['drug', 'route'],
    columns='scenario',
    values='risk_quotient'
).round(3)

print('Risk Quotient Summary (RQ = MEC_effluent / PNEC)')
print('RQ > 1.0 = HIGH risk | RQ 0.1-1.0 = MODERATE | RQ < 0.1 = LOW')
print()
rq_summary

---
## 5. Visualizations
*Placeholder — full visualizations built in Day 4*

In [ ]:
# Quick RQ bar chart — preview only
baseline = daily_loads[daily_loads['scenario'] == 'baseline'].copy()
baseline['drug_route'] = baseline['drug'] + ' (' + baseline['route'] + ')'

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if rq >= 1 else '#ff7f0e' if rq >= 0.1 else '#2ca02c'
          for rq in baseline['risk_quotient']]

ax.barh(baseline['drug_route'], np.log10(baseline['risk_quotient'] + 1e-6), color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--', label='RQ = 1 threshold (log scale)')
ax.set_xlabel('log10(Risk Quotient)')
ax.set_title('GLP-1 Risk Quotients — Baseline Scenario (2027)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/rq_preview.png', dpi=150)
plt.show()
print('✓ Preview chart saved → data/rq_preview.png')

---
## 6. Next Steps (Day 3)

- [ ] Uncertainty analysis — Monte Carlo simulation on excretion fractions and WWTP removal rates
- [ ] Temporal modeling — project RQ growth from 2024 → 2030 as adoption scales
- [ ] Geographic concentration — model RQ variation by city population and WWTP capacity
- [ ] Organic waste stream shift — model reduced caloric load from GLP-1-driven weight loss
- [ ] Agricultural demand change — downstream fertilizer and land use implications